# Expanded weed species training — local files, temporary Colab GPU
Open this local notebook through Colab File > Upload notebook. Select the existing T4 runtime and Run All. The first cell asks for one file: `training/verified_v2_colab_inputs.zip`. All data stays in temporary `/content`; no Google Drive is mounted or used by this workflow.
Each epoch uses every designated training image, with bounded minority replay. Validation alone controls early stopping and calibration. The untouched test set is evaluated after training. Checkpoints are offered for download every five completed epochs; keep the browser open and allow the requested downloads. A checkpoint is safe from runtime deletion only after it is downloaded.


In [ ]:
import hashlib, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then reconnect. CPU training is disabled.'
a=torch.arange(16,device='cuda',dtype=torch.float32).reshape(4,4)
assert (a@a.T).sum().item()==3680.0
print('Verified GPU:',torch.cuda.get_device_name(0))
from google.colab import files
inputs=Path('/content')/'verified_v2_colab_inputs.zip'
if not inputs.is_file():
    print('Choose the local training/verified_v2_colab_inputs.zip file. Upload goes to temporary runtime storage only.')
    files.upload()
assert inputs.is_file(), 'Select verified_v2_colab_inputs.zip'
with inputs.open('rb') as source:assert hashlib.file_digest(source,'sha256').hexdigest()=='62bb3a087521f7183e8b9b8324bcbef3f13640c8e8e452366fe22bbd2deed667', 'Input archive SHA256 mismatch'
destination=Path('/content/weed_expansion')
with zipfile.ZipFile(inputs) as archive:
    assert archive.testzip() is None
    for entry in archive.infolist():assert (destination/entry.filename).resolve().is_relative_to(destination)
    archive.extractall(destination)
print('Input upload verified')


In [ ]:
import json, subprocess, sys
from pathlib import Path
project = Path('/content/weed_expansion')
pid_file = project/'expansion_pid.json'
if pid_file.exists():
    pid = json.loads(pid_file.read_text())['pid']
    command = Path(f'/proc/{pid}/cmdline')
    if command.exists() and b'scripts.remote_expansion_job' in command.read_bytes():
        raise RuntimeError(f'Existing worker {pid} is running. Use the monitor cell to follow it.')
status_file = project/'evaluation/expansion_job.json'
if status_file.exists() and json.loads(status_file.read_text())['state'] == 'complete':
    raise RuntimeError('This runtime already has a completed export. Use the monitor cell to download it.')
with (project/'expansion.log').open('a') as log:
    worker = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.remote_expansion_job', '--verified-combined'],
                              cwd=project, stdout=log, stderr=subprocess.STDOUT)
pid_file.write_text(json.dumps({'pid': worker.pid}))
print('Worker launched. The epoch log will confirm training progress.')


In [ ]:
import hashlib, json, time
from pathlib import Path
from google.colab import files
root=Path('/content/weed_expansion')
sent=set();previous=None
while True:
    status_path=root/'evaluation/expansion_job.json'
    if not status_path.exists():time.sleep(2);continue
    try:status=json.loads(status_path.read_text())
    except json.JSONDecodeError:time.sleep(2);continue
    progress_path=root/'models/v2/training_status.json'
    if progress_path.exists():
        try:progress=json.loads(progress_path.read_text())
        except json.JSONDecodeError:progress={}
    else:progress={}
    marker=(status['state'],progress.get('epochs'))
    if marker!=previous:
        print(json.dumps({'job':status,'training':progress}),flush=True);previous=marker
    # Completed immutable snapshots survive concurrent training writes. These are
    # protected against runtime loss only after the browser has downloaded them.
    for checkpoint in sorted((root/'models/v2/recovery').glob('epoch_*.pt')):
        if checkpoint.name not in sent:
            with checkpoint.open('rb') as source:digest=hashlib.file_digest(source,'sha256').hexdigest()
            print('Recovery snapshot ready:',checkpoint.name,'SHA256',digest,flush=True)
            files.download(str(checkpoint));sent.add(checkpoint.name)
    if status['state']=='error':raise RuntimeError(status['traceback'])
    if status['state']=='complete':
        archive=root/'expanded_artifacts.zip'
        with archive.open('rb') as source:digest=hashlib.file_digest(source,'sha256').hexdigest()
        print('Training and evaluation completed. Export SHA256:',digest,flush=True)
        print('Save the archive and transfer receipt inside your local project. Runtime storage remains temporary.')
        files.download(str(root/'evaluation/expansion_transfer.json'))
        files.download(str(archive))
        break
    time.sleep(30)
